模型（model）是智能体Agent的大脑，负责推理分析。工具（tools）是Agent的手脚，负责执行任务，与外界交互。
用户提问（input）
|
model-模型分析：-是否需要调用工具
|             -调用哪个工具
|             -工具执行结果是否足以回答用户问题
|工具-调用工具
    -分析工具执行结果（将执行结果返回给模型）
|
输出output

模型、工具是核心

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

一个完整的agent至少包含2个关键的部分
-模型：是agent的大脑，负责推理、分析、规划任务步骤
-工具:是agent的手脚，负责执行任务，与外界交互
因此，定义带工具的agent的基本流程给下：
-定义工具
-初始化模型（可以省略，有的可以在创建智能体时初始化）
-初始化agent，绑定模型和工具

自定义工具
所谓的工具，本质是一个可用的函数，但是这个函数不是我们自己去调用，而是给模型调用，因此除了定义函数外，我们还需要描述这个工具，让模型知道这个工具如何使用，包括下列信息：
-工具名
-工具的作用
-工具需要的参数

我们的工具定义在本地，而模型是在云端的，他是怎么知道我们本地的工具是怎么使用的呢
我们的智能体拿到用户问题时不会直接发给模型，而是先把本地所有的工具的信息做一个组装（将工具的名字、工具作用的描述、工具需要的参数等）都封装起来。然后和用户的问题一起组装成一个参数发给模型，模型这样知道本地有什么工具他们的作用是什么，同时分析是否需要调用工具以及调用什么工具

In [2]:
#1、描述工具的方式：基于tool描述工具（基于装饰器传参）
#langchain中定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用
def tool(x:float) -> float:#参数是浮点数
    return x**0.5#求参数/任意数的平方根
#他只是函数，不是工具，因为缺少工具名字、描述、需要的参数

In [3]:
from langchain.tools import tool#传入工具的名字（专业一点）、交代工具的作用,简答的业务描述
@tool("square_root",description="calculate the square root of a number")
def square_root(x:float) -> float:
    return x**0.5
#总结：顺序是先定义函数，然后在上面加tool装饰器（制定工具名字以及详细的描述）

In [4]:
#2、使用函数名和文档注释来描述工具
"""
如果我们不传/@tool装饰器没有定义工具名和作用的描述，此时：
-工具名：默认是函数名
-工具所需的参数：默认就是函数的参数列表
-工具作用的描述：默认就是函数的文档注释
-同时不用加tool装饰器的参数
"""
@tool
def square_root(x:float) -> float:
    """
    calculate the square root of a number
    """
    return x**0.5

In [5]:
#定义一个查询天气的tool,location地址，units单位，include_forecast是否包含天气预报///参数很多业务也是很复杂的
#使用args来描述参数。
#工具的作用，工具的每一个参数的作用
#推荐使用默认函数名+函数文档的这种写法
@tool
def get_weather(location:str,units:str="celsius",include_forecast:bool= False) -> str:
    """
    Get current weather and optionally forecast.
    Arg:
        location:city name or coordinates
        units:unit of degrees
        include_forecast:does it include the weather forecast
    """
    temp=22 if units=="celsius" else 72
    result=f"current weather in {location}:{temp} degrees {units[0].upper()}"
    if include_forecast:
        result+="\nNext 5 days :Sunny"
    return result


In [9]:
#对于特殊参数，列表的描述可能不方便，例如：units单位是枚举类型，要么是华氏度，要么是摄氏度。
#使用pydanticmodel来描述参数（适用于参数复杂时）
from pydantic import BaseModel,Field#field是字段
from typing import Literal#导入用到的类型，例如枚举
#都是先指定类型(例如字符串str，后面接=Field（字段的描述，可以加默认值）
#；例如一个查天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location:str=Field(description="City name or coordinates")
    units:Literal["celsius","fahrenheit"]=Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast:bool=Field(
        default=False,
        description="Include 5-day forecast"
    )
#定义一个查询天气的tool,使用args_schema的参数/参数约束来使用刚才定义的模型,将模型信息传给装饰器，langchain就能把这些信息提取出来作为参数的描述,再结合函数里的文档注释的描述，使langchain直观了解工具的信息
@tool(args_schema=WeatherInput)
def get_weather(location:str,units:str="celsius",include_forecast:bool=False) -> str:
    """
    Get current weather and optionally forecast.
    """
    temp=22 if units=="celsius" else 72
    result=f"current weather in {location}:{temp} degrees {units[0].upper()}"
    if include_forecast:
        result+="\nNext 5 days :Sunny"
    return result


In [7]:
#工具调用方式与普通函数调用方式一致,invoke调用，不是直接传参，因此传入的是一个字典
square_root.invoke({"x":467})

21.61018278497431

In [10]:
get_weather.invoke({"location":"London","units":"celsius","include_forecast":True})
#直观看工具效果是使用工具.invoke调用

'current weather in London:22 degrees C\nNext 5 days :Sunny'

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
#将工具传给智能体
agent = create_agent(model="deepseek-chat",
                     tools=[square_root,get_weather],)

In [12]:
for token,metadata in agent.stream(
        {"messages":[HumanMessage("扬州接下来几天天气如何？")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

我需要知道您希望使用哪种温度单位来查看扬州的天气预报。您希望使用摄氏度（celsius）还是华氏度（fahrenheit）呢？

另外，我会为您获取扬州接下来几天的天气预报信息。current weather in 扬州:22 degrees C
Next 5 days :Sunny根据天气预报，扬州当前的天气是22摄氏度，接下来5天的天气预计都是晴朗的。

看起来天气信息比较简洁，可能没有提供详细的每日预报数据。不过从现有信息来看，扬州接下来几天天气不错，温度适宜，都是晴天，适合外出活动。

In [13]:
response=agent.invoke(
    {"messages":[HumanMessage("457和13的平方根是多少？")]}
)
print(response)

{'messages': [HumanMessage(content='457和13的平方根是多少？', additional_kwargs={}, response_metadata={}, id='a82594df-7815-48dd-882d-f768eff1baf5'), AIMessage(content='我来帮您计算这两个数的平方根。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 425, 'total_tokens': 476, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 41}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '3be905f4-ef3c-443f-bb77-4dc30620500b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daa15-02ec-7930-8e2a-de91d7f125a1-0', tool_calls=[{'name': 'square_root', 'args': {'x': 457}, 'id': 'call_00_PhuSbknINbfMx47dztqKbsMe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 425, 'output_tokens': 51, 'total_tokens': 47

In [14]:
for message in response["messages"]:
    print(message.pretty_print())

================================ Human Message =================================

457和13的平方根是多少？
None
================================== Ai Message ==================================

我来帮您计算这两个数的平方根。
Tool Calls:
  square_root (call_00_PhuSbknINbfMx47dztqKbsMe)
 Call ID: call_00_PhuSbknINbfMx47dztqKbsMe
  Args:
    x: 457
None
================================= Tool Message =================================
Name: square_root

21.37755832643195
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_00_AqSusOWMWrZdQx7rs0qmFlaO)
 Call ID: call_00_AqSusOWMWrZdQx7rs0qmFlaO
  Args:
    x: 13
None
================================= Tool Message =================================
Name: square_root

3.605551275463989
None
================================== Ai Message ==================================

计算结果如下：

- 457的平方根 ≈ 21.3776
- 13的平方根 ≈ 3.6056
None


使用2个toolcalls,返回2个工具调用，我们本地模型拿到这2个后（智能体分析要调用这两个就去调用），并且得到2个结果，根据这2个结果6再组织一下，模型根据这个结果回答就行了